In [ ]:
import pandas as pd
import numpy as np
import joblib

# --- 1. 定義必要的數學函數  ---
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exps = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

def forward_pass(X, w1, b1, w2, b2):
    n1 = X @ w1 + b1
    a1 = relu(n1) 
    n2 = a1 @ w2 + b2
    a2 = softmax(n2)
    return a2

def encode_labels(df):
    labels = np.zeros((df.shape[0], 4))
    for i in range(df.shape[0]):
        val = df.iloc[i]['DIQ010']
        if val == 1.0: 
            labels[i] = [0, 0, 0, 1]  # 有糖尿病
        elif val == 2.0: 
            labels[i] = [0, 0, 1, 0]  # 無糖尿病
        elif val == 3.0: 
            labels[i] = [0, 1, 0, 0]  # 邊緣性
        elif val == 9.0: 
            labels[i] = [1, 0, 0, 0]  # 無法回答
    return labels

In [ ]:
# --- 2. 載入模型參數與 Scaler ---
# 載入權重 
params = np.load('./output/best_model_params.npz')

w1 = params['w1']
b1 = params['b1']

w2 = params['w2']
b2 = params['b2']

# 載入預處理器 
loaded_scaler = joblib.load('./output/scaler.joblib')

In [ ]:
# --- 3. 載入並處理新數據集 ---
new_data = pd.read_csv('./data_set/Nhanes_cleaned_15_16.csv')

# 移除序列號 
new_data_cleaned = new_data.drop(columns=['SEQN'])

Y_true = encode_labels(new_data_cleaned)
X_new_raw = new_data_cleaned.drop(columns=['DIQ010'])

# 使用加載的 Scaler 進行標準化 
X_new_scaled = loaded_scaler.transform(X_new_raw)

In [ ]:
# --- 4. 執行預測與準確率計算 ---
# 進行順向傳播
Y1_pred_probs = forward_pass(X_new_scaled, w1, b1, w2, b2)

# 取得預測類別索引
predicted_classes = np.argmax(Y1_pred_probs, axis=1)
actual_classes = np.argmax(Y_true, axis=1)

# 計算準確率 
accuracy = np.mean(predicted_classes == actual_classes)

print(f"新數據集的預測準確率: {accuracy * 100:.2f}%")

新數據集的預測準確率: 91.93%
